# Chat COCO Reference Images

In [ ]:
!pip install -q transformers torch torchvision pillow accelerate huggingface-hub requests DeepCache torch-fidelity torchmetrics
!pip show torch transformers accelerate sentencepiece


In [ ]:
import os
import json
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from diffusers import StableDiffusionPipeline, DiffusionPipeline
from torch_fidelity import calculate_metrics
from DeepCache import DeepCacheSDHelper
from PIL import Image
import torch.nn.functional as F
from transformers import CLIPProcessor, CLIPModel

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [ ]:
!pip freeze > requirements.txt

In [ ]:


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

BATCH_SIZE = 8 if DEVICE == "cuda" else 2
NUM_STEPS = 25
NUM_IMAGES = 1000   # important for stable FID

OUTPUT_ROOT = "outputs"
COCO_DIR = "coco_real"

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [ ]:
!wget http://images.cocodataset.org/zips/val2017.zip
!wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip

!unzip val2017.zip
!unzip annotations_trainval2017.zip

Streaming output truncated to the last 5000 lines.
 extracting: val2017/000000365745.jpg  
 extracting: val2017/000000320425.jpg  
 extracting: val2017/000000481404.jpg  
 extracting: val2017/000000314294.jpg  
 extracting: val2017/000000335328.jpg  
 extracting: val2017/000000513688.jpg  
 extracting: val2017/000000158548.jpg  
 extracting: val2017/000000132116.jpg  
 extracting: val2017/000000415238.jpg  
 extracting: val2017/000000321333.jpg  
 extracting: val2017/000000081738.jpg  
 extracting: val2017/000000577584.jpg  
 extracting: val2017/000000346905.jpg  
 extracting: val2017/000000433980.jpg  
 extracting: val2017/000000228144.jpg  
 extracting: val2017/000000041872.jpg  
 extracting: val2017/000000117492.jpg  
 extracting: val2017/000000368900.jpg  
 extracting: val2017/000000376900.jpg  
 extracting: val2017/000000352491.jpg  
 extracting: val2017/000000330790.jpg  
 extracting: val2017/000000384850.jpg  
 extracting: val2017/000000032735.jpg  
 extracting: val2017/00000019

In [ ]:


COCO_IMG_DIR = "val2017"
COCO_ANN_FILE = "annotations/captions_val2017.json"


def load_coco_val2017():
    with open(COCO_ANN_FILE, "r") as f:
        data = json.load(f)

    # Map image_id -> captions
    captions_dict = {}
    for ann in data["annotations"]:
        img_id = ann["image_id"]
        captions_dict.setdefault(img_id, []).append(ann["caption"])

    # Map image_id -> filename
    id_to_file = {img["id"]: img["file_name"] for img in data["images"]}

    samples = []
    for img_id, file_name in id_to_file.items():
        path = os.path.join(COCO_IMG_DIR, file_name)
        captions = captions_dict.get(img_id, [])

        if os.path.exists(path) and captions:
            samples.append({
                "image_path": path,
                "caption": captions[0]  # pick first caption
            })

    return samples


samples = load_coco_val2017()
print(f"Loaded {len(samples)} samples")

Loaded 5000 samples


In [ ]:
OUTPUT_REAL = "coco_real"
os.makedirs(OUTPUT_REAL, exist_ok=True)

for i, sample in enumerate(samples):
    img = Image.open(sample["image_path"]).convert("RGB")
    img = img.resize((512, 512))
    img.save(f"{OUTPUT_REAL}/{i}.png")

print("Saved resized COCO images")



Saved resized COCO images


In [ ]:
def load_prompts(samples, num_images):
    return [s["caption"] for s in samples[:num_images]]


prompts = load_prompts(samples, NUM_IMAGES)

def load_pipeline(name):
    if name == "sd1.5":
        pipe = StableDiffusionPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=DTYPE
        )
    elif name == "sd2.1":
        pipe = DiffusionPipeline.from_pretrained(
            "sd2-community/stable-diffusion-2-1", #"stabilityai/stable-diffusion-2-1-base",
            torch_dtype=DTYPE
        )

    pipe = pipe.to(DEVICE)
    pipe.set_progress_bar_config(disable=True)
    return pipe

def enable_cache(pipe, use_cache):
    if not use_cache:
        return None

    helper = DeepCacheSDHelper(pipe)
    helper.set_params(cache_interval=5)  # better quality
    helper.enable()
    return helper

In [ ]:
def generate_images(pipe, prompts, out_dir, use_cache):
    os.makedirs(out_dir, exist_ok=True)

    helper = enable_cache(pipe, use_cache)

    all_times = []

    for i in tqdm(range(0, len(prompts), BATCH_SIZE)):
        batch_prompts = prompts[i:i+BATCH_SIZE]
        batch_size = len(batch_prompts)

        # SAME LATENTS for fairness
        latents = torch.randn(
            (batch_size, pipe.unet.in_channels, 64, 64),
            device=DEVICE,
            dtype=DTYPE
        )

        start = time.time()

        images = pipe(
            batch_prompts,
            num_inference_steps=NUM_STEPS,
            latents=latents
        ).images

        end = time.time()
        all_times.append(end - start)

        for j, img in enumerate(images):
            img.save(f"{out_dir}/{i+j}.png")

    if helper:
        helper.disable()

    return np.mean(all_times), np.std(all_times)

def compute_fid(real_dir, fake_dir):
    metrics = calculate_metrics(
        input1=real_dir,
        input2=fake_dir,
        fid=True,
        isc=False,
        kid=False,
        cuda=(DEVICE == "cuda")
    )
    return metrics["frechet_inception_distance"]

In [ ]:
NUM_IMAGES = len(samples)
prompts = load_prompts(samples, NUM_IMAGES)

configs = [
    ("sd1.5", False),
    ("sd1.5", True),
    ("sd2.1", False),
    ("sd2.1", True),
]

results = {}

for model, cache in configs:
    print(f"\nRunning {model} | cache={cache}")

    pipe = load_pipeline(model)

    out_dir = f"{OUTPUT_ROOT}/{model}_{'cache' if cache else 'base'}"

    mean_t, std_t = generate_images(pipe, prompts, out_dir, cache)

    results[(model, cache)] = {
        "dir": out_dir,
        "time_mean": mean_t,
        "time_std": std_t
    }

    del pipe
    torch.cuda.empty_cache()


Running sd1.5 | cache=False


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

...

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--sd2-community--stable-diffusion-2-1/snapshots/bb2154823665391b4fb29b0b9cf82a198964ee05/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 1/625 [00:12<2:08:33, 12.36s/it]

  0%|          | 2/625 [00:24<2:10:01, 12.52s/it]

  0%|          | 3/625 [00:37<2:11:00, 12.64s/it]

  1%|          | 4/625 [00:50<2:10:26, 12.60s/it]

  1%|          | 5/625 [01:02<2:09:40, 12.55s/it]

  1%|          | 6/625 [01:15<2:09:00, 12.51s/it]

  1%|          | 7/625 [01:27<2:08:10, 12.44s/it]

  1%|▏         | 8/625 [01:39<2:07:41, 12.42s/it]

  1%|▏         | 9/625 [01:52<2:08:04, 12.48s/it]

  2%|▏         | 10/625 [02:04<2:08:00, 12.49s


Running sd2.1 | cache=True


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--sd2-community--stable-diffusion-2-1/snapshots/bb2154823665391b4fb29b0b9cf82a198964ee05/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 1/625 [00:05<52:09,  5.01s/it]

  0%|          | 2/625 [00:10<52:06,  5.02s/it]

  0%|          | 3/625 [00:14<51:22,  4.96s/it]

  1%|          | 4/625 [00:19<51:07,  4.94s/it]

  1%|          | 5/625 [00:24<50:59,  4.94s/it]

  1%|          | 6/625 [00:29<51:06,  4.95s/it]

  1%|          | 7/625 [00:34<50:38,  4.92s/it]

  1%|▏         | 8/625 [00:39<50:24,  4.90s/it]

  1%|▏         | 9/625 [00:44<50:36,  4.93s/it]

  2%|▏         | 10/625 [00:49<50:37,  4.94s/it]

  2%|▏        

In [ ]:
fid_results = {}

for model in ["sd1.5", "sd2.1"]:
    base_dir = results[(model, False)]["dir"]
    cache_dir = results[(model, True)]["dir"]

    fid_base = compute_fid(COCO_DIR, base_dir)
    fid_cache = compute_fid(COCO_DIR, cache_dir)

    fid_results[model] = {
        "base": fid_base,
        "cache": fid_cache,
        "delta": fid_cache - fid_base
    }

print("\n=== FINAL RESULTS ===")

for (model, cache), data in results.items():
    print(f"{model} | cache={cache}")
    print(f"  Time: {data['time_mean']:.3f} ± {data['time_std']:.3f}")

for model, vals in fid_results.items():
    print(f"\n{model} FID:")
    print(f"  Base:  {vals['base']:.3f}")
    print(f"  Cache: {vals['cache']:.3f}")
    print(f"  ΔFID:  {vals['delta']:.3f}")

Creating feature extractor "inception-v3-compat" with features ['2048']
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth


  0%|          | 0.00/91.2M [00:00<?, ?B/s]

 11%|█         | 10.1M/91.2M [00:00<00:01, 84.2MB/s]

 22%|██▏       | 20.1M/91.2M [00:00<00:00, 93.1MB/s]

 33%|███▎      | 30.1M/91.2M [00:00<00:00, 98.0MB/s]

 44%|████▍     | 40.1M/91.2M [00:00<00:00, 80.9MB/s]

 60%|██████    | 55.1M/91.2M [00:00<00:00, 104MB/s] 

 73%|███████▎  | 66.6M/91.2M [00:00<00:00, 109MB/s]

 85%|████████▌ | 77.6M/91.2M [00:00<00:00, 111MB/s]

100%|██████████| 91.2M/91.2M [00:00<00:00, 97.0MB/s]
Extracting statistics from input 1
Looking for samples non-recursivelty in "coco_real" with extensions png,jpg,jpeg
Found 5000 samples


Processing samples:   0%|          | 0/5000 [00:00<?, ?samples/s]

Processing samples:   1%|▏         | 64/5000 


=== FINAL RESULTS ===
sd1.5 | cache=False
  Time: 12.775 ± 0.062
sd1.5 | cache=True
  Time: 4.851 ± 0.009
sd2.1 | cache=False
  Time: 11.570 ± 0.024
sd2.1 | cache=True
  Time: 4.047 ± 0.011

sd1.5 FID:
  Base:  25.330
  Cache: 24.489
  ΔFID:  -0.841

sd2.1 FID:
  Base:  29.861
  Cache: 42.028
  ΔFID:  12.167


Frechet Inception Distance: 42.02799


### CLIP Computation

In [ ]:


def load_clip():
    model = CLIPModel.from_pretrained(
        "openai/clip-vit-base-patch32"
    ).to(DEVICE)

    processor = CLIPProcessor.from_pretrained(
        "openai/clip-vit-base-patch32"
    )

    model.eval()
    return model, processor

In [ ]:

def compute_clip_score(image_dir, prompts, model, processor):
    image_files = sorted(os.listdir(image_dir))

    scores = []

    for i in tqdm(range(0, len(image_files), BATCH_SIZE)):
        batch_files = image_files[i:i+BATCH_SIZE]
        batch_images = []
        batch_texts = []

        for j, file in enumerate(batch_files):
            img = Image.open(os.path.join(image_dir, file)).convert("RGB")
            batch_images.append(img)

            # match prompt index
            idx = i + j
            batch_texts.append(prompts[idx])

        inputs = processor(
            text=batch_texts,
            images=batch_images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=77
        ).to(DEVICE)

        with torch.no_grad():
            outputs = model(**inputs)

            image_embeds = outputs.image_embeds
            text_embeds = outputs.text_embeds

            # Normalize (CRITICAL)
            image_embeds = F.normalize(image_embeds, dim=-1)
            text_embeds = F.normalize(text_embeds, dim=-1)

            # cosine similarity
            batch_scores = (image_embeds * text_embeds).sum(dim=-1)

        scores.extend(batch_scores.cpu().numpy())

    return float(np.mean(scores))

In [ ]:
clip_model, clip_processor = load_clip()

clip_results = {}

for model in ["sd1.5", "sd2.1"]:
    base_dir = results[(model, False)]["dir"]
    cache_dir = results[(model, True)]["dir"]

    print(f"\nComputing CLIP for {model}...")

    clip_base = compute_clip_score(
        base_dir, prompts, clip_model, clip_processor
    )

    clip_cache = compute_clip_score(
        cache_dir, prompts, clip_model, clip_processor
    )

    clip_results[model] = {
        "base": clip_base,
        "cache": clip_cache,
        "delta": clip_cache - clip_base
    }

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]


Computing CLIP for sd1.5...




  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 1/625 [00:00<04:03,  2.56it/s]

  0%|          | 2/625 [00:00<02:29,  4.16it/s]

  0%|          | 3/625 [00:00<01:58,  5.24it/s]

  1%|          | 4/625 [00:00<01:42,  6.03it/s]

  1%|          | 5/625 [00:00<01:34,  6.56it/s]

  1%|          | 6/625 [00:01<01:32,  6.71it/s]

  1%|          | 7/625 [00:01<01:28,  7.01it/s]

  1%|▏         | 8/625 [00:01<01:25,  7.25it/s]

  1%|▏         | 9/625 [00:01<01:23,  7.39it/s]

  2%|▏         | 10/625 [00:01<01:21,  7.54it/s]

  2%|▏         | 11/625 [00:01<01:20,  7.61it/s]

  2%|▏         | 12/625 [00:01<01:20,  7.59it/s]

  2%|▏         | 13/625 [00:01<01:19,  7.68it/s]

  2%|▏         | 14/625 [00:02<01:21,  7.54it/s]

  2%|▏         | 15/625 [00:02<01:20,  7.56it/s]

  3%|▎         | 16/625 [00:02<01:19,  7.67it/s]

  3%|▎         | 17/625 [00:02<01:19,  7.67it/s]

  3%|▎         | 18/625 [00:02<01:19,  7.62it/s]

  3%|▎         | 19/625 [00:02<01:17,  7.78it/s]

  3%|▎         |


Computing CLIP for sd2.1...


100%|██████████| 625/625 [01:17<00:00,  8.11it/s]


# Results

In [ ]:
for model, vals in clip_results.items():
    print(f"\n{model} CLIP:")
    print(f"  Base:  {vals['base']:.4f}")
    print(f"  Cache: {vals['cache']:.4f}")
    print(f"  ΔCLIP: {vals['delta']:.4f}")


sd1.5 CLIP:
  Base:  0.1419
  Cache: 0.1452
  ΔCLIP: 0.0033

sd2.1 CLIP:
  Base:  0.1479
  Cache: 0.1554
  ΔCLIP: 0.0075


| Model | Speedup | FID ↑ | CLIP Δ |
| ----- | ------- | ----- | ------ |
| SD1.5 | 2.63×    | -0.841   | 0.0033  |
| SD2.1 | 2.86×    | 12.167  | 0.0075  |

| Model Name, Cache Setting | Training Time | Mean Time | Variance Time | FID Score| CLIP Score
|---|---|---|---|---|---|
| SD1.5, Cache=False | 2 hrs 21 mins 51 secs |12.775 seconds | 0.062 seconds | 25.330 | 0.1419
| SD1.5, Cache=True | 59 mins 32 secs |4.851 seconds | 0.009 seconds | 24.489 | 0.1452
| SD2.1, Cache=False | 2 hrs 9 min 42 secs|11.570 seconds | 0.024 seconds | 29.861 | 0.1479
| SD2.1, Cache=True | 51 mins 30 secs |4.047 seconds | 0.011 seconds | 42.028 | 0.1554

In [ ]:
"""
sd1.5 | cache=False
  Time: 12.775 ± 0.062
sd1.5 | cache=True
  Time: 4.851 ± 0.009
sd2.1 | cache=False
  Time: 11.570 ± 0.024
sd2.1 | cache=True
  Time: 4.047 ± 0.011

sd1.5 FID:
  Base:  25.330
  Cache: 24.489
  ΔFID:  -0.841

sd2.1 FID:
  Base:  29.861
  Cache: 42.028
  ΔFID:  12.167
"""

In [3]:
# Code to Fix Github notebook renderer error
# nvalid Notebook There was an error rendering your Notebook: the 'state' key is missing from 'metadata.widgets'. Add 'state' to each, or remove 'metadata.widgets'. Using nbformat v5.10.4 and nbconvert v7.16.6
import json
filepath = "CAP6614_Final_Proj_DeepCache_SD15_SD21.ipynb"
with open(filepath, "r", encoding="utf-8") as f:
    nb = json.load(f)
if "metadata" in nb and "widgets" in nb["metadata"]:
    del nb["metadata"]["widgets"]
with open(filepath, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=2) 